<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks/3.2-feature-extraction-with-pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Generando tensores mediante el uso de Pytoch - Torchaudio

In [5]:
# Imports
import os
import torch
from torch.utils.data import Dataset
import tqdm as tqdm
import pandas as pd
import torchaudio
import torchaudio.functional as F
import torchaudio.transforms as T
import kagglehub
import shutil

In [2]:
# Descargarmos directamente desde Kaggle
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
path_crema = kagglehub.dataset_download("ejlok1/cremad")

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Using Colab cache for faster access to the 'cremad' dataset.


In [3]:
# Variables de configuración y rutas
#--------------------------------------------------------------------------
FAST_ROOT_DIR = '/content/datasets'
os.makedirs(FAST_ROOT_DIR, exist_ok=True)
# Config
SAMPLE_RATE = 16000 # Limitado por CREMA
MIN_DURATION = 0.5
MAX_DURATION = 3
TARGET_SAMPLES = (SAMPLE_RATE * MAX_DURATION )
HOP_LENGTH = 512
# Cálculo matemático del ancho del tensor (Frames)
TARGET_FRAMES = int((SAMPLE_RATE * MAX_DURATION) / HOP_LENGTH) + 1 # ~94 frames
PAD_MODE = "constant" # Para padding de audios cortos
# Rutas Faster
#-------------------------------------------------------------------------
OUT_DIR_PY_TENSORS = '/content/split_pytorch_tensors'
os.makedirs(OUT_DIR_PY_TENSORS, exist_ok=True)

In [4]:
# Copiamos a local(session) para mayor velocidad en el procesamiento (SSD)
!cp -r {path}/* {FAST_ROOT_DIR}
!cp -r {path_crema}/* {FAST_ROOT_DIR}

In [6]:
# Eliminamos los archivos duplicados dentro de ravdess
from genericpath import isdir
target_dir = os.path.join(FAST_ROOT_DIR, "audio_speech_actors_01-24")

if os.path.isdir(target_dir):
  shutil.rmtree(target_dir)
  print("Directorio eliminado")
else:
  ("Directorio no encontrado")

Directorio eliminado


In [ ]:
def get_actor_and_emotion(filename):
  """Extrae Actor ID y Emoción basándose en la nomenclatura estricta de RAVDESS/CREMA-D"""
  if filename.startswith('03'): # Nomenclatura Ravdess (ej. 03-02-XX-01-01-02.wav)
    parts = filename.split('-')
    if len(parts) != 7: return 'unknown'
    actor_id = f"ravdess_{parts[-1].replace('.wav', '')}" # En Python, los índices negativos cuentan desde el final hacia atrás
    rav_map = {1:'neutral', 2:'neutral', 3:'happy', 4:'sad', 5:'angry', 6:'fearful', 7:'disgust', 8:'surprised'}
    emotion = rav_map.get(int(parts[2]), 'unknown')
    return actor_id, emotion

  else: # Nomenclatura CREMA-D (ej. 1001_DFA_ANG_XX.wav)
      parts = filename.split('_')
      if len(parts) < 3: return None, 'unknown'
      actor_id = f"crema_{parts[0]}"
      crema_map = {'NEU':'neutral', 'HAP':'happy', 'SAD':'sad', 'ANG':'angry', 'FEA':'fearful', 'DIS':'disgust'}
      emotion = crema_map.get(parts[2].upper(), 'unknown')
      return actor_id, emotion


# 1. Mapeo de archivos físicos (aqui se mapearian las copias si no las eliminamos)
all_files = [os.path.join(root, f) for root, dirs, files in os.walk(FAST_ROOT_DIR) for f in files if f.endswith('.wav')]
actor_to_files = {}

for f in all_files:
    actor, emo = get_actor_and_emotion(os.path.basename(f))
    if emo == 'unknown': continue
    if actor not in actor_to_files: actor_to_files[actor] = []
    actor_to_files[actor].append(f)

# 2. Partición estocástica de Actores (80% Train, 10% Val, 10% Test)
unique_actors = __builtins__.list(actor_to_files.keys())
random.seed(42) # Semilla inmutable para consistencia entre ejecuciones
random.shuffle(unique_actors)

train_split = int(0.8 * len(unique_actors))
val_split = int(0.9 * len(unique_actors))

actor_splits = {}
for a in unique_actors[:train_split]: actor_splits[a] = 'train'
for a in unique_actors[train_split:val_split]: actor_splits[a] = 'val'
for a in unique_actors[val_split:]: actor_splits[a] = 'test'

print(f"Total Actores: {len(unique_actors)} | Train: {train_split} | Val: {val_split - train_split} | Test: {len(unique_actors) - val_split}")


In [ ]:
class AudioFeatureExtractor(torch.nn.Module):
  def __init__(self, sr=16000, n_mels=60):
    self.sr= sr
    self.mel_spec = T